# Zadanie 3.1c: ręczny dobór parametrów ODE i porównanie PDE a `ODE_1` a `ODE_2`

Cel (issue #20): dobrać ręcznie i przejrzyście fizyczne parametry surogatów ($K_{\text{eff}}, c, \gamma$)
bez optymalizatora czarnej skrzynki, a następnie porównać błędy $M(t)$, $M_{\min}$ i $t_{\text{odrostu}}$
pomiędzy pełnym PDE, `ODE_1` i `ODE_2`:
1. na czterech scenariuszach z Zadania 2.2 (#8),
2. na mapie niedopasowania $(r, s)$ z #14.

## Reguła ręcznego doboru
- **$c = c_0 = \frac{\int R(x)u_0(x)\,dx}{\int u_0(x)\,dx}$** — geometryczny udział masy guza w wiązce w chwili $t = 0$ (bez dopasowywania).
- **$K_{\text{eff}}$** — efektywna pojemność środowiska dla horyzontu $t \in [0, 10]$; w PDE szerokość efektywna guza rośnie od $W_{\text{eff}}(0) \approx 14{,}4$ do ok. $46$ w $t = 10$, więc sprawdzamy ręczną siatkę kilku okrągłych wartości $K_{\text{eff}} \in \{25, 30, 35\}$.
- **$\gamma = \gamma_0 D$** — tempo wymiany dyfuzyjnej przez brzeg wiązki; sprawdzamy ręcznie $\gamma_0 \in \{0{,}4,\;0{,}8,\;1{,}2,\;1{,}6\}$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, grid, initial_condition
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.ode import ODE1Params, ODE2Params, initial_beam_fraction, simulate_ode1, simulate_ode2
from offtarget.solvers import simulate

rng = seed_everything()

x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
u0 = initial_condition(x)
M0 = float(tumor_mass(u0, x))
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)

plans = {
    "pole pokrywa guz, duża dawka": Params(beta=1.5, s=0.0, r=20.0),
    "pole pokrywa guz, mała dawka": Params(beta=0.45, s=0.0, r=20.0),
    "pole za wąskie": Params(beta=1.5, s=0.0, r=3.0),
    "pole przesunięte": Params(beta=1.5, s=6.0, r=8.0),
}
pde_M = {name: tumor_mass(simulate(p, schedule, x, t, method="cn"), x) for name, p in plans.items()}
c0_map = {name: initial_beam_fraction(x, u0, p) for name, p in plans.items()}

## 1. Ręczny przegląd siatki $(K_{\text{eff}}, \gamma_0)$ na scenariuszach z #8

In [ ]:
grid_rows = []
for K_eff in (25.0, 30.0, 35.0):
    for g0 in (0.4, 0.8, 1.2, 1.6):
        errs = []
        for name, p in plans.items():
            c0 = c0_map[name]
            mi, mo = simulate_ode2(
                c0 * M0, (1.0 - c0) * M0,
                ODE2Params(rho=p.rho, K_eff=K_eff, beta=p.beta, c=c0, gamma=g0 * p.D),
                schedule, t,
            )
            M_hat = mi + mo
            errs.append(float(np.linalg.norm(M_hat - pde_M[name]) / np.linalg.norm(pde_M[name])))
        grid_rows.append({
            "K_eff": K_eff,
            "gamma_0": g0,
            "sredni_blad_L2": round(float(np.mean(errs)), 3),
            "maks_blad_L2": round(float(np.max(errs)), 3),
        })

df_grid = pd.DataFrame(grid_rows)
print(df_grid.to_string(index=False))

# Wybrana ręcznie para kompromisowa dla całego horyzontu [0, 10]:
K_EFF_CHOSEN = 30.0
GAMMA0_CHOSEN = 1.2

## 2. Tabela porównawcza: PDE a `ODE_1` a `ODE_2` (scenariusze z #8)

In [ ]:
comp_rows = []
traj = {}
for name, p in plans.items():
    c0 = c0_map[name]
    M_p = pde_M[name]
    M_1 = simulate_ode1(M0, ODE1Params(rho=p.rho, K_eff=K_EFF_CHOSEN, beta=p.beta, c=c0), schedule, t)
    mi, mo = simulate_ode2(
        c0 * M0, (1.0 - c0) * M0,
        ODE2Params(rho=p.rho, K_eff=K_EFF_CHOSEN, beta=p.beta, c=c0, gamma=GAMMA0_CHOSEN * p.D),
        schedule, t,
    )
    M_2 = mi + mo
    traj[name] = (M_p, M_1, M_2)

    mmin_p, _ = min_mass(t, M_p)
    mmin_1, _ = min_mass(t, M_1)
    mmin_2, _ = min_mass(t, M_2)
    treg_p = time_to_regrowth(t, M_p)
    treg_1 = time_to_regrowth(t, M_1)
    treg_2 = time_to_regrowth(t, M_2)

    comp_rows.append({
        "scenariusz": name,
        "c0": round(c0, 3),
        "M_min_PDE": round(mmin_p, 3),
        "M_min_ODE1": round(mmin_1, 3),
        "M_min_ODE2": round(mmin_2, 3),
        "t_odrostu_PDE": round(treg_p, 2) if np.isfinite(treg_p) else np.nan,
        "t_odrostu_ODE1": round(treg_1, 2) if np.isfinite(treg_1) else np.nan,
        "t_odrostu_ODE2": round(treg_2, 2) if np.isfinite(treg_2) else np.nan,
        "blad_L2_ODE1": round(float(np.linalg.norm(M_1 - M_p) / np.linalg.norm(M_p)), 3),
        "blad_L2_ODE2": round(float(np.linalg.norm(M_2 - M_p) / np.linalg.norm(M_p)), 3),
        "MAE_ODE1": round(float(np.mean(np.abs(M_1 - M_p))), 2),
        "MAE_ODE2": round(float(np.mean(np.abs(M_2 - M_p))), 2),
    })

df_comp = pd.DataFrame(comp_rows)
df_comp.to_csv(TABLES / "zad3_01c_ode_porownanie.csv", index=False)
print("\nPorównanie na scenariuszach z #8:")
print(df_comp.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.5), sharex=True, sharey=True)
for ax, (name, (M_p, M_1, M_2)) in zip(axes.flat, traj.items()):
    ax.plot(t, M_p, color="black", lw=2.2, label="PDE (CN)")
    ax.plot(t, M_1, color="tab:blue", ls="--", lw=1.8, label="ODE_1")
    ax.plot(t, M_2, color="tab:orange", lw=2.0, label="ODE_2")
    for ti in schedule.times:
        ax.axvline(ti, color="tab:red", alpha=0.14, lw=0.8)
    ax.axhline(M0, color="gray", ls=":", lw=0.9)
    ax.set_title(f"{name} ($c_0={c0_map[name]:.2f}$)")
    ax.grid(alpha=0.25)
axes[0, 0].legend(frameon=False)
for ax in axes[1]:
    ax.set_xlabel("czas $t$")
for ax in axes[:, 0]:
    ax.set_ylabel("masa guza $M(t)$")
fig.suptitle(f"PDE vs ODE_1 vs ODE_2 (ręcznie: $K_{{eff}}={K_EFF_CHOSEN:.0f}$, $\\gamma={GAMMA0_CHOSEN:.1f}\\,D$)", y=0.99)
fig.tight_layout()
fig.savefig(FIGURES / "zad3_01c_ode_scenariusze.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## 3. Błąd na mapie niedopasowania $(r, s)$ z #14

Liczymy siatkę $18 \times 18$ po $r \in [1, 24]$ i $s \in [0, 15]$ przy $\beta = 1{,}5$ ($\Delta x = 0{,}2$, `cn`)
i porównujemy mapy błędów względnych $L^2$ oraz błędu $|M_{\min}^{\text{ODE}} - M_{\min}^{\text{PDE}}|$ dla `ODE_1` i `ODE_2`.

In [ ]:
x_m, _ = grid(80.0, 400)
u0_m = initial_condition(x_m)
M0_m = float(tumor_mass(u0_m, x_m))

r_vals = np.linspace(1.0, 24.0, 18)
s_vals = np.linspace(0.0, 15.0, 18)

err1_l2 = np.zeros((len(s_vals), len(r_vals)))
err2_l2 = np.zeros((len(s_vals), len(r_vals)))
err1_min = np.zeros((len(s_vals), len(r_vals)))
err2_min = np.zeros((len(s_vals), len(r_vals)))

map_rows = []
for i, sv in enumerate(s_vals):
    for j, rv in enumerate(r_vals):
        p = Params(beta=1.5, s=float(sv), r=float(rv))
        M_p = tumor_mass(simulate(p, schedule, x_m, t, method="cn", dt_max=0.05), x_m)
        c0 = initial_beam_fraction(x_m, u0_m, p)
        M_1 = simulate_ode1(M0_m, ODE1Params(rho=p.rho, K_eff=K_EFF_CHOSEN, beta=p.beta, c=c0), schedule, t)
        mi, mo = simulate_ode2(
            c0 * M0_m, (1.0 - c0) * M0_m,
            ODE2Params(rho=p.rho, K_eff=K_EFF_CHOSEN, beta=p.beta, c=c0, gamma=GAMMA0_CHOSEN * p.D),
            schedule, t,
        )
        M_2 = mi + mo

        norm_p = float(np.linalg.norm(M_p))
        e1 = float(np.linalg.norm(M_1 - M_p) / norm_p)
        e2 = float(np.linalg.norm(M_2 - M_p) / norm_p)
        dm1 = abs(float(M_1.min()) - float(M_p.min()))
        dm2 = abs(float(M_2.min()) - float(M_p.min()))
        err1_l2[i, j], err2_l2[i, j] = e1, e2
        err1_min[i, j], err2_min[i, j] = dm1, dm2

        map_rows.append({
            "s": round(float(sv), 3),
            "r": round(float(rv), 3),
            "c0": round(c0, 4),
            "M_min_PDE": round(float(M_p.min()), 4),
            "M_min_ODE1": round(float(M_1.min()), 4),
            "M_min_ODE2": round(float(M_2.min()), 4),
            "blad_L2_ODE1": round(e1, 4),
            "blad_L2_ODE2": round(e2, 4),
        })

df_map = pd.DataFrame(map_rows)
df_map.to_csv(TABLES / "zad3_01c_ode_mapa.csv", index=False)

summary_map = pd.DataFrame([
    {
        "model": "ODE_1",
        "mediana_L2": round(float(np.median(err1_l2)), 3),
        "srednia_L2": round(float(np.mean(err1_l2)), 3),
        "p90_L2": round(float(np.percentile(err1_l2, 90)), 3),
        "sredni_blad_M_min": round(float(np.mean(err1_min)), 3),
    },
    {
        "model": "ODE_2",
        "mediana_L2": round(float(np.median(err2_l2)), 3),
        "srednia_L2": round(float(np.mean(err2_l2)), 3),
        "p90_L2": round(float(np.percentile(err2_l2, 90)), 3),
        "sredni_blad_M_min": round(float(np.mean(err2_min)), 3),
    },
])
print("\nPodsumowanie błędów na całej mapie (r, s):")
print(summary_map.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2), sharex=True, sharey=True)
extent = [r_vals[0], r_vals[-1], s_vals[0], s_vals[-1]]
vmax = float(max(np.percentile(err1_l2, 95), np.percentile(err2_l2, 95)))

for ax, Z, title in [
    (axes[0], err1_l2, "Względny błąd $L^2$: ODE_1"),
    (axes[1], err2_l2, "Względny błąd $L^2$: ODE_2"),
]:
    im = ax.imshow(Z, origin="lower", aspect="auto", extent=extent, cmap="magma_r", vmin=0.0, vmax=vmax)
    ax.plot(s_vals + 6.0, s_vals, color="cyan", ls="--", lw=1.4, label="$r = s + 6$")
    ax.set_title(title)
    ax.set_xlabel("połowa szerokości wiązki $r$")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

diff = err1_l2 - err2_l2
dlim = float(np.max(np.abs(diff)))
im_d = axes[2].imshow(diff, origin="lower", aspect="auto", extent=extent, cmap="RdBu_r", vmin=-dlim, vmax=dlim)
axes[2].plot(s_vals + 6.0, s_vals, color="black", ls="--", lw=1.4, label="$r = s + 6$")
axes[2].set_title("Poprawa (błąd ODE_1 $-$ ODE_2)\nczerwony = wygrywa ODE_2")
axes[2].set_xlabel("połowa szerokości wiązki $r$")
fig.colorbar(im_d, ax=axes[2], fraction=0.046, pad=0.04)

axes[0].set_ylabel("przesunięcie wiązki $s$")
axes[0].legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.8)
fig.tight_layout()
fig.savefig(FIGURES / "zad3_01c_ode_mapa.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## Wnioski

- **W strefie pełnego pokrycia ($r \ge s + 6$, $c_0 \approx 1$):** `ODE_1` i `ODE_2` są praktycznie równoważne i osiągają mały błąd (ok. 18–23% względnego $L^2$ przy jednej globalnej stałej $K_{\text{eff}} = 30$).
- **W strefie niedopasowania ($r < s + 6$, $0 < c_0 < 1$):** `ODE_2` wyraźnie wygrywa z `ODE_1` na całej mapie $(r, s)$ — zarówno pod względem trajektorii $M(t)$, jak i minimum masy $M_{\min}$ oraz czasu odrostu $t_{\text{odrostu}}$.
- **Ograniczenie obu surogatów ODE:** stała pojemność $K_{\text{eff}}$ nie oddaje liniowego poszerzania się fali Fishera w czasie (w PDE guz bez leczenia rośnie z $12$ do $45{,}8$, bo oba fronty wędrują w nieograniczoną tkankę). Dlatego przy $K_{\text{eff}} = 30$ w scenariuszu „pole za wąskie” masa końcowa nasyca się wcześniej niż w PDE.
- **Wąski pas tuż pod przekątną ($r \approx s + 8 \dots s + 12$):** nawet w `ODE_2` zostaje tam podwyższony błąd, bo w chwili $t = 0$ guz mieści się w wiązce ($c_0 \approx 0{,}99$), ale w PDE front Fishera zdąży zdyfundować poza brzeg wiązki jeszcze przed pierwszą frakcją i między frakcjami — efektu tego nie da się opisać samą stałą $c_0$ bez informacji przestrzennej.